# 1〜9 を順に並べて `+`, `*`, `( )` で作れる数をすべて列挙する

**ルール**
- 数字 1,2,3,4,5,6,7,8,9 をこの順に並べる
- 隣り合う数字はつなげて多桁の数にしてよい（例：`12`, `456`, `89`）
- 演算は `+` と `*` のみ。かっこは入れ子でも自由に使ってよい
- 例：`12+3*(4+56)+7+89`

（つなげる操作は**数字どうし**だけで、`(1+2)3` のように計算結果をつなげることはできません。）

**方針**：区間DP（動的計画法）で、すべての式を漏れなく調べます。しくみは次の節で説明します。

## 区間DPのしくみ

区間DPは、並んだものの**区間ごとに答えを求めて表に記録し、短い区間から長い区間へ順に組み立てていく**動的計画法です。

### 1. 式は「最後の演算」で必ず2つに分かれる

どんな式でも、いちばん最後に行う演算（式の木の根）に注目すると、数字の列は**左の連続した区間**と**右の連続した区間**に分かれます。

```
12 + 3*(4+56) + 7 + 89
最後の + で分けると  [1〜7 の部分] + [8〜9 の部分]
```

数字の順番は変えられないので、左側は必ず $d_i \cdots d_k$、右側は必ず $d_{k+1} \cdots d_j$ という区間になります。
左右のそれぞれも同じように最後の演算で分かれるので、式は**二分木**として表せます。かっこの入れ子は、この木の形がそのまま表しています。

### 2. 漸化式

$S(i,j)$ を「数字 $d_i, \ldots, d_j$ から作れる値の集合」とすると

$$
S(i,j) = \underbrace{\{\, d_i d_{i+1}\cdots d_j \,\}}_{\text{全部つなげた数}} \;\cup\; \bigcup_{k=i}^{j-1} \underbrace{\{\, a+b,\ a\times b \mid a\in S(i,k),\ b\in S(k+1,j) \,\}}_{\text{区切り位置 } k \text{ で左右に分けて結合}}
$$

区切り位置 $k$ をすべて試すことで、すべての式（すべての二分木）を漏れなく調べられます。

### 3. 小さな例：1, 2, 3

| 区間 | 計算 | 値の集合 |
|---|---|---|
| $S(1,1)$ | `1` | $\{1\}$ |
| $S(2,2)$ | `2` | $\{2\}$ |
| $S(3,3)$ | `3` | $\{3\}$ |
| $S(1,2)$ | `12`, `1+2`, `1*2` | $\{2, 3, 12\}$ |
| $S(2,3)$ | `23`, `2+3`, `2*3` | $\{5, 6, 23\}$ |
| $S(1,3)$ | `123` ／ $k=1$: $\{1\}$ と $S(2,3)$ を結合 → 6, 23, 7, 6, 24, 23 ／ $k=2$: $S(1,2)$ と $\{3\}$ を結合 → 5, 6, 6, 9, 15, 36 | $\{5, 6, 7, 9, 15, 23, 24, 36, 123\}$ |

$S(1,3)$ は **9種類**になります。

### 4. 計算の順番と速さの理由

- **短い区間から順に計算する**：区間の長さ `len` を外側のループにすると、$S(i,j)$ を作る時点で、それより短い $S(i,k)$ と $S(k+1,j)$ がもう完成しています。
- **表に記録して使い回す**：たとえば $S(2,3)$ は $S(1,3)$ でも $S(2,4)$ でも使います。区間は 1〜9 で $9 \times 10 / 2 = 45$ 個しかありません。
- **値を集合（辞書）で持つ**：式が違っても値が同じなら1つにまとめるので、組み合わせの数が爆発しにくくなります。1〜9 の式は約19万通りありますが、値は約4万3千種類です。

この3つのおかげで、全探索でも一瞬で終わります。

In [1]:
# 各区間について「値 => (その値を作る式の一例, 式のいちばん外側の演算)」を記録する
# 外側の演算: 'n' = 数そのもの, '+' = 和, '*' = 積
function allexpressions(d::Vector{Int})
    n = length(d)
    S = [Dict{Int,Tuple{String,Char}}() for i in 1:n, j in 1:n]
    for len in 1:n, i in 1:n-len+1
        j = i + len - 1
        D = S[i, j]
        # 数字をそのままつなげた数
        num = foldl((a, b) -> 10a + b, d[i:j])
        D[num] = (string(num), 'n')
        # 区切り位置 k で左右に分けて + / * で結合
        for k in i:j-1, (a, (ea, oa)) in S[i, k], (b, (eb, ob)) in S[k+1, j]
            haskey(D, a + b) || (D[a + b] = (ea * "+" * eb, '+'))
            if !haskey(D, a * b)
                pa = oa == '+' ? "(" * ea * ")" : ea   # 和を掛けるときだけかっこが必要
                pb = ob == '+' ? "(" * eb * ")" : eb
                D[a * b] = (pa * "*" * pb, '*')
            end
        end
    end
    return S[1, n]
end

allexpressions (generic function with 1 method)

### 小さな例で表の中身を見る

上の `allexpressions` は値ごとに式の例も記録していますが、しくみを見るために**値の集合だけ**を持つ簡単な版で、1, 2, 3 の表 $S(i,j)$ を短い区間から順に表示してみます。
上の「小さな例」の表と一致するはずです。

In [2]:
# 1,2,3 で表 S(i,j) の中身を確認する（短い区間から順に）
function intervaltable(d::Vector{Int})
    n = length(d)
    S = [Set{Int}() for i in 1:n, j in 1:n]
    for len in 1:n, i in 1:n-len+1
        j = i + len - 1
        push!(S[i, j], foldl((a, b) -> 10a + b, d[i:j]))   # 全部つなげた数
        for k in i:j-1, a in S[i, k], b in S[k+1, j]          # 区切り位置 k
            push!(S[i, j], a + b, a * b)
        end
        println("S(", i, ",", j, ") = ", sort!(collect(S[i, j])))
    end
end

intervaltable([1, 2, 3])

S(1,1) = [1]
S(2,2) = [2]
S(3,3) = [3]
S(1,2) = [2, 3, 12]
S(2,3) = [5, 6, 23]
S(1,3) = [5, 6, 7, 9, 15, 23, 24, 36, 123]


In [3]:
@time R = allexpressions(collect(1:9))
vals = sort!(collect(keys(R)))
println("作れる数の個数: ", length(vals))
println("最小値: ", first(vals), "   ", R[first(vals)][1])
println("最大値: ", last(vals), "   ", R[last(vals)][1])

  0.150837 seconds (403.83 k allocations: 26.103 MiB, 9.92% gc time, 94.69% compilation time)
作れる数の個数: 43350
最小値: 44   1*(2+3+4+5+6+7+8+9)
最大値: 123456789   123456789


## 同じ値になる式は何通りあるか

同じ値になる式はたくさんあります（例：`1+2+3` と `1*2*3` はどちらも 6）。
そこで値ごとに式が何通りあるかも数えます。

ただし `(1+2)+3` と `1+(2+3)` のように、**結合法則でかっこの位置が違うだけの式は同じもの**とみなします。
そのために「`+` の右側にくる式は `+` を外側にもたない」「`*` の右側にくる式は `*` を外側にもたない」という
左結合の形（例：`1+2+3` は `(1+2)+3` の形）だけを数えます。
これで、余分なかっこを外したときに見た目が異なる式をちょうど1回ずつ数えられます。

例：1〜3 なら `123, 1+23, 1*23, 12+3, 12*3, 1+2+3, 1*2*3, 1+2*3, 1*2+3, (1+2)*3, 1*(2+3)` の 11 通り。

In [4]:
# C[i,j][v] = [数そのもの, 外側が+, 外側が*] の式の個数
function countexpressions(d::Vector{Int})
    n = length(d)
    C = [Dict{Int,Vector{Int}}() for i in 1:n, j in 1:n]
    for len in 1:n, i in 1:n-len+1
        j = i + len - 1
        D = C[i, j]
        num = foldl((a, b) -> 10a + b, d[i:j])
        get!(D, num, zeros(Int, 3))[1] += 1
        for k in i:j-1, (a, ca) in C[i, k], (b, cb) in C[k+1, j]
            L = sum(ca)
            get!(D, a + b, zeros(Int, 3))[2] += L * (cb[1] + cb[3])   # 右側は外側が + でない式
            get!(D, a * b, zeros(Int, 3))[3] += L * (cb[1] + cb[2])   # 右側は外側が * でない式
        end
    end
    return Dict(v => sum(c) for (v, c) in C[1, n])
end

println("1〜3 の式の数: ", sum(values(countexpressions([1, 2, 3]))))  # 11 になるはず
N = countexpressions(collect(1:9))
println("式の総数: ", sum(values(N)))
println("式が1通りしかない値: ", count(==(1), values(N)), " 個")

1〜3 の式の数: 11
式の総数: 190399
式が1通りしかない値: 21459 個


## 式が多い値ランキング

In [5]:
for (v, c) in sort(collect(N), by = x -> -x[2])[1:10]
    println(lpad(v, 6), " : ", lpad(c, 4), " 通り   例 ", R[v][1])
end

   945 :  116 通り   例 1*(2+3)*(45+6*(7+8+9))
   864 :  109 通り   例 1*2*3*(45+6*(7+8)+9)
   163 :  105 通り   例 1*(2+3+4+5*(6+7)+89)
   164 :  100 通り   例 1+2+3+4+5*(6+7)+89
   729 :   96 通り   例 1+2*(3*4+5*67+8+9)
   594 :   95 通り   例 1*(2+3+4+(5+67)*8+9)
  1080 :   95 通り   例 1+2+3*(4+5*(6+7*8+9))
   737 :   95 通り   例 1+2*(3+4*5+6*7*8+9)
   801 :   94 通り   例 1*(2+34+((5+6)*7+8)*9)
   981 :   94 通り   例 1*(2+3+4+(5*6+78)*9)


## 作れる数の一覧（小さい順、式の一例と式の数）

In [6]:
for v in vals[1:100]
    println(lpad(v, 6), " = ", rpad(R[v][1], 32), lpad(N[v], 4), " 通り")
end

    44 = 1*(2+3+4+5+6+7+8+9)                8 通り
    45 = 1+2+3+4+5+6+7+8+9                  8 通り
    46 = 1+2*3+4+5+6+7+8+9                  1 通り
    48 = (1+2)*3+4+5+6+7+8+9                1 通り
    49 = 1*(2+3*4+5+6+7+8+9)               13 通り
    50 = 1+2+3*4+5+6+7+8+9                  2 通り
    54 = 1*(2*(3+4+5)+6+7+8+9)              6 通り
    55 = 1*(2+3+4*5+6+7+8+9)               15 通り
    56 = 1+2+3+4*5+6+7+8+9                  9 通り
    57 = 1+2*3+4*5+6+7+8+9                  1 通り
    59 = 1*(2+3*(4+5)+6+7+8+9)             15 通り
    60 = 1*(2*(3+4+5+6)+7+8+9)              6 通り
    61 = 1+2*(3+4+5+6)+7+8+9                1 通り
    62 = 1*(23+4+5+6+7+8+9)                 7 通り
    63 = 1+23+4+5+6+7+8+9                   9 通り
    64 = 1+2+3+4+5*6+7+8+9                 12 通り
    65 = 1+2*3+4+5*6+7+8+9                  3 通り
    66 = (1+2)*(3+4+5)+6+7+8+9              1 通り
    67 = 1*(2+(3+4)*5+6+7+8+9)             10 通り
    68 = 1+2+(3+4)*5+6+7+8+9               13 通り
    69 = 1+2+3*4+5*6

## 作れない数

最小値から 1000 までのうち、作れない正の整数を調べます。

In [7]:
valset = Set(vals)
missing_nums = [m for m in 1:1000 if m ∉ valset]
println("1〜1000 で作れない数 (", length(missing_nums), " 個):")
println(missing_nums)

1〜1000 で作れない数 (48 個):
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 47, 51, 52, 53, 58]


## 検算

求めた式の文字列を Julia に実際に計算させて、値が一致するか確認します。

In [8]:
ok = all(v -> eval(Meta.parse(R[v][1])) == v, vals)
println("すべての式が正しい: ", ok)
# 式が 1〜9 をこの順に使っているかも確認
println("数字の並びが 123456789: ", all(v -> filter(isdigit, R[v][1]) == "123456789", vals))

すべての式が正しい: true
数字の並びが 123456789: true


## 結果をファイルに保存

すべての値・式の一例・式の数を CSV に書き出します。

In [9]:
open("enzan_1to9.csv", "w") do io
    println(io, "value,expression,count")
    for v in vals
        println(io, v, ",", R[v][1], ",", N[v])
    end
end
println("enzan_1to9.csv に ", length(vals), " 行を書き出しました")

enzan_1to9.csv に 43350 行を書き出しました


## おまけ：数字の個数を変えたとき

1〜n（n = 1,…,9）で作れる値の種類と式の数の変化。

In [10]:
for n in 1:9
    Nn = countexpressions(collect(1:n))
    println("1〜", n, ": 値 ", lpad(length(Nn), 5), " 種類,  式 ", lpad(sum(values(Nn)), 6), " 通り")
end

1〜1: 値     1 種類,  式      1 通り
1〜2: 値     3 種類,  式      3 通り
1〜3: 値     9 種類,  式     11 通り
1〜4: 値    32 種類,  式     47 通り
1〜5: 値   133 種類,  式    223 通り
1〜6: 値   532 種類,  式   1135 通り
1〜7: 値  2285 種類,  式   6063 通り
1〜8: 値  9874 種類,  式  33535 通り
1〜9: 値 43350 種類,  式 190399 通り
